# 🎬 Kanji: ảnh câu chuyện → video hoạt hình (Wan 2.2, mã nguồn mở)

Model **Wan2.2-TI2V-5B** (Alibaba, Apache 2.0) — ảnh + câu mô tả → clip ~5 giây, 1280×704, 24 fps.

**Cần GPU ≥ 24 GB** cho nhanh: *Runtime → Change runtime type →* **L4** hoặc **A100** (Colab Pro).
GPU T4 miễn phí (16 GB) vẫn thử được nhờ chế độ tiết kiệm bộ nhớ nhưng rất chậm (có thể 30–60 phút/clip) hoặc hết bộ nhớ.

Chạy lần lượt từng ô từ trên xuống. Kết quả: `kanji_clips.zip` (tải về máy).


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Cài Wan 2.2 + tải model (~20 GB, mất 5–10 phút)

In [ ]:
%cd /content
!git clone -q https://github.com/Wan-Video/Wan2.2.git
%cd /content/Wan2.2
# cài mọi thứ trừ flash_attn (tự biên dịch trên Colab rất lâu / hay lỗi)
!grep -v -i "flash" requirements.txt > /tmp/req.txt && pip -q install -r /tmp/req.txt
# thử flash_attn bản dựng sẵn; lỗi thì bỏ qua (Wan dùng attention của PyTorch, chậm hơn chút)
!pip -q install flash-attn --no-build-isolation --only-binary=:all: || echo "⚠️ Không cài được flash_attn — vẫn chạy được, chỉ chậm hơn"
!pip -q install "huggingface_hub[cli]"
!huggingface-cli download Wan-AI/Wan2.2-TI2V-5B --local-dir /content/Wan2.2-TI2V-5B

## 2. Chọn thẻ + câu mô tả

`số thẻ: "mô tả chuyển động"` — viết tiếng Anh cho model hiểu tốt nhất. Giữ câu *simple hand-drawn cartoon, white background, static camera* để không đổi nét vẽ.
Ảnh lấy từ trang web: `https://testervippro.github.io/Giai-Thich-Ngu-Phap-Tieng-Anh/kanji-viewer/kanji_story/NNNN.webp`


In [ ]:
STYLE = "simple hand-drawn cartoon line art, flat colors, white background, static camera, gentle smooth motion, keep the same drawing style"
CARDS = {
    2095: "A bald Buddhist monk with closed eyes gently rubs his shiny bald head with one hand and smiles peacefully, small golden sparkles twinkle above his head",
    # 1051: "...",   # thêm thẻ khác ở đây
}
SITE = "https://testervippro.github.io/Giai-Thich-Ngu-Phap-Tieng-Anh/kanji-viewer/kanji_story/"

## 3. Chuẩn bị ảnh: cắt phần tranh (bên trái ảnh câu chuyện), đặt giữa khung 1280×704 nền trắng

In [ ]:
import io, os, urllib.request
import numpy as np
from PIL import Image

os.makedirs("/content/in", exist_ok=True)

def picture_part(im):
    """Tranh ở bên trái; cắt tới khoảng cột trắng đầu tiên (rộng ≥ 12 px) sau 25% chiều rộng."""
    a = np.asarray(im.convert("RGB")).astype(int); h, w, _ = a.shape
    nonwhite = (a.min(axis=2) < 235).mean(axis=0)
    run = 0
    for x in range(int(w * .25), int(w * .7)):
        run = run + 1 if nonwhite[x] < .01 else 0
        if run >= 12:
            return im.crop((0, 0, x - run + 1, h))
    return im.crop((0, 0, int(w * .42), h))

for no in CARDS:
    raw = urllib.request.urlopen(SITE + f"{no:04d}.webp").read()
    pic = picture_part(Image.open(io.BytesIO(raw)).convert("RGB"))
    W, H = 1280, 704
    s = min(W * .9 / pic.width, H * .9 / pic.height)
    pic = pic.resize((round(pic.width * s), round(pic.height * s)), Image.LANCZOS)
    canvas = Image.new("RGB", (W, H), "white")
    canvas.paste(pic, ((W - pic.width) // 2, (H - pic.height) // 2))
    canvas.save(f"/content/in/{no:04d}.png")
    display(canvas.resize((640, 352)))

## 4. Tạo video (mỗi thẻ ~3–6 phút trên A100/L4)

In [ ]:
import subprocess, os
os.makedirs("/content/out", exist_ok=True)
%cd /content/Wan2.2
for no, motion in CARDS.items():
    out = f"/content/out/{no:04d}.mp4"
    if os.path.exists(out):
        continue
    cmd = ["python", "generate.py", "--task", "ti2v-5B", "--size", "1280*704",
           "--ckpt_dir", "/content/Wan2.2-TI2V-5B", "--offload_model", "True", "--convert_model_dtype", "--t5_cpu",
           "--image", f"/content/in/{no:04d}.png", "--prompt", f"{motion}. {STYLE}", "--save_file", out]
    print(" ".join(cmd[:4]), "...", no)
    subprocess.run(cmd, check=True)
print(os.listdir("/content/out"))

## 5. Xem + tải về

In [ ]:
from IPython.display import Video, display
for f in sorted(os.listdir("/content/out")):
    display(Video(f"/content/out/{f}", embed=True, width=640))
!cd /content/out && zip -q -r /content/kanji_clips.zip .
from google.colab import files
files.download("/content/kanji_clips.zip")